# LiciGuard · Preparar los datos

Este cuaderno se ejecuta **una vez** (o cuando agregues datos nuevos). Prepara lo que la app necesita:

| Parte | Qué hace | Resultado |
|---|---|---|
| A | Corta la normativa (Ley 32069, Reglamento, bases estándar) en artículos | `data/normativa/articulos.json` |
| B | Prueba la extracción sobre **tus** bases y crea una hoja para revisarla a mano | `data/bases/revision.csv` |
| C | Filtra los datos abiertos de contrataciones de tu rubro | `data/ocds/historial.csv` |
| D | Mide qué tan bien acierta el sistema (después de revisar la hoja de la parte B) | Porcentaje de aciertos |

**Cómo se usa:** ejecuta cada celda de arriba hacia abajo con `Shift + Enter`. Si una celda da error, cópialo y pregúntame.

> Todo esto funciona **sin IA y sin gastar tokens**.

## Paso 0 · Solo si usas Google Colab

Si abriste este cuaderno en **Colab**, ejecuta las 2 celdas siguientes. En tu computadora (VS Code) puedes ejecutarlas igual: no hacen nada.

1. La primera te pedirá subir el archivo **liciguard-ai.zip** (el proyecto completo).
2. La segunda te deja subir tus PDF (bases y normativa).

In [ ]:
# Paso 0.1 · Preparar Colab (sube liciguard-ai.zip cuando aparezca el botón)
import sys, os, subprocess, zipfile
EN_COLAB = "google.colab" in sys.modules
if EN_COLAB:
    from google.colab import files
    if not os.path.exists("/content/liciguard-ai/core"):
        print("Sube el archivo liciguard-ai.zip 👇")
        for nombre in files.upload():
            if nombre.endswith(".zip"):
                zipfile.ZipFile(nombre).extractall("/content")
    os.chdir("/content/liciguard-ai")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pymupdf"], check=True)
    print("Listo. Proyecto en", os.getcwd())
else:
    print("Estás en tu computadora: no hace falta este paso.")

In [ ]:
# Paso 0.2 · (Colab) Subir tus PDF. Cambia CARPETA y vuelve a ejecutar para cada tipo.
CARPETA = "bases"        # "bases" para las bases de licitaciones · "normativa" para la ley y el reglamento
if EN_COLAB:
    destino = os.path.join("data", CARPETA)
    os.makedirs(destino, exist_ok=True)
    for nombre, contenido in files.upload().items():
        with open(os.path.join(destino, nombre), "wb") as f:
            f.write(contenido)
        os.remove(nombre) if os.path.exists(nombre) else None
    print(f"En data/{CARPETA}:", os.listdir(destino))

In [ ]:
# 0) Preparación: ubica la carpeta del proyecto y carga las herramientas
import sys, json
from pathlib import Path

RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(RAIZ))

import pandas as pd
from core.config import ARCHIVO_NORMATIVA, ARCHIVO_HISTORIAL
from core.pdf_parser import extraer_paginas
from core.preguntas import analizar
from core.buscador import Normativa
from core.preparar import preparar_normativa, preparar_historial

pd.set_option("display.max_colwidth", 120)
print("Carpeta del proyecto:", RAIZ)

## Parte A · Normativa → artículos

1. Guarda los PDF oficiales en `data/normativa/` (Ley 32069, Reglamento actualizado, bases estándar, opiniones).
2. Escribe abajo, para cada archivo, **de dónde lo bajaste** (fuente y enlace). Así cada cita de la app será verificable.
3. Ejecuta la celda.

In [ ]:
FUENTES = {
    # "nombre_del_archivo.pdf": {"fuente": "Nombre que verá el usuario", "url": "enlace oficial"},
    "ley_32069.pdf": {"fuente": "Ley N.° 32069", "url": "https://www.gob.pe/institucion/oece/colecciones/45029-ley-n-32069-ley-general-de"},
    "reglamento_ley_32069.pdf": {"fuente": "Reglamento de la Ley N.° 32069 (D.S. 009-2025-EF y modif.)", "url": None},
    "bases_estandar_bienes.pdf": {"fuente": "Bases estándar OECE · bienes", "url": None},
}

carpeta = RAIZ / "data" / "normativa"
print("PDF encontrados:", [p.name for p in carpeta.glob("*.pdf")])
articulos = preparar_normativa(carpeta, FUENTES, ARCHIVO_NORMATIVA)
print(f"Artículos guardados: {len(articulos)} → {ARCHIVO_NORMATIVA}")
if articulos:
    display(pd.DataFrame(articulos)[["fuente", "articulo", "titulo"]].head(10))
else:
    print("⚠️ No hay PDF en data/normativa/. Súbelos (Paso 0.2 con CARPETA = 'normativa') y vuelve a ejecutar.")

In [ ]:
# Prueba el buscador: ¿encuentra los artículos correctos?
norm = Normativa.cargar()
for consulta in ["requerimiento marca equivalente", "experiencia del postor monto facturado", "penalidad por mora"]:
    print("\n🔎", consulta)
    for a in norm.buscar(consulta, k=2):
        print(f"   {a['fuente']} · {a['articulo']} (puntaje {a['puntaje']}): {a['texto'][:110]}…")

## Parte B · Probar la extracción en tus bases

1. Guarda tus PDF de bases en `data/bases/`.
2. Ajusta el perfil de ejemplo si quieres.
3. Ejecuta: verás qué encontró el sistema en cada expediente, con su página y cita.

In [ ]:
PERFIL_DEMO = {"experiencia_soles": 320000, "certificaciones": []}
norm = Normativa.cargar()

filas = []
for pdf in sorted((RAIZ / "data" / "bases").glob("*.pdf")):
    paginas = extraer_paginas(pdf.read_bytes())
    r = analizar(paginas, PERFIL_DEMO, norm)
    d = r["datos"]
    print(f"\n📄 {pdf.name} · {len(paginas)} págs · {d.get('entidad')} · valor estimado {d.get('valor_estimado')}")
    print(f"   Recomendación: {r['recomendacion']['titulo']}")
    for q in r["requisitos"]:
        filas.append({"archivo": pdf.name, "tipo": "requisito", "categoria": q["categoria"], "valor": q["valor"],
                      "pagina": q["pagina"], "cita": q["cita"], "correcto (si/no)": ""})
    for h in r["justa"]["hallazgos"]:
        filas.append({"archivo": pdf.name, "tipo": "señal", "categoria": h["severidad"], "valor": h["titulo"],
                      "pagina": h["pagina"], "cita": h["cita"], "correcto (si/no)": ""})

if not filas:
    print("⚠️ No hay PDF en data/bases/. Súbelos (Paso 0.2 con CARPETA = 'bases') y vuelve a ejecutar.")
revision = pd.DataFrame(filas)
ruta_rev = RAIZ / "data" / "bases" / "revision.csv"
if not revision.empty:
    revision.to_csv(ruta_rev, index=False, encoding="utf-8-sig")
    print(f"\nHoja de revisión creada: {ruta_rev}")
revision.head(20)

**Tarea manual (es tu "harness" de evaluación):** abre `data/bases/revision.csv` en Excel, lee cada fila y escribe **si** o **no** en la columna *correcto*.
Además anota en una fila nueva cualquier requisito importante que el sistema **no** encontró (pon `faltó` en *correcto*). Luego ve a la Parte D.

## Parte C · Datos abiertos → historial de tu rubro

1. Descarga de https://data.open-contracting.org/en/publication/135 los archivos **JSON** (`2025.jsonl.gz`, `2026.jsonl.gz`).
2. Ponlos en `data/ocds/` **sin descomprimir**.
3. Escribe las palabras de tu rubro. Primero prueba con `LIMITE = 50_000` (rápido); si todo sale bien, pon `LIMITE = None` para leer todo (puede tardar varios minutos).

In [ ]:
# (Opcional) Descargar los datos abiertos directamente, sin pasar por tu computadora.
# Recomendado en Colab. Los archivos pesan bastante: puede tardar varios minutos.
import urllib.request
AÑOS = ["2026"]          # agrega "2025" si quieres más historia
DESCARGAR = EN_COLAB     # en tu computadora déjalo así y descárgalos desde el navegador; o pon True
if DESCARGAR:
    os.makedirs("data/ocds", exist_ok=True)
    for año in AÑOS:
        url = f"https://data.open-contracting.org/en/publication/135/download?name={año}.jsonl.gz"
        destino = f"data/ocds/{año}.jsonl.gz"
        if not os.path.exists(destino):
            print("Descargando", url)
            urllib.request.urlretrieve(url, destino)
        print(destino, round(os.path.getsize(destino) / 1e6), "MB")

In [ ]:
PALABRAS = ["uniforme", "prenda", "vestuario", "confeccion", "textil", "chaleco", "polo", "camisa", "pantalon"]
ENTIDADES = []          # opcional: ["municipalidad de ...", "hospital ..."] para quedarte solo con esas
LIMITE = 50_000         # None = leer todo

archivos = sorted((RAIZ / "data" / "ocds").glob("*.jsonl*"))
print("Archivos:", [a.name for a in archivos])
n = preparar_historial(archivos, PALABRAS, ARCHIVO_HISTORIAL, ENTIDADES, limite_lineas=LIMITE)
print(f"Procesos guardados: {n:,} → {ARCHIVO_HISTORIAL}")

In [ ]:
# Mira lo que se guardó
hist = pd.read_csv(ARCHIVO_HISTORIAL)
print(len(hist), "procesos")
print("\nEntidades con más compras de tu rubro:")
print(hist["entidad"].value_counts().head(10))
print("\nProveedores que más ganan:")
print(hist["ganador"].value_counts().head(10))
hist.head(10)

## Parte D · ¿Qué tan bien acierta? (después de revisar la hoja)

Esta métrica es la que puedes mostrar al jurado: **es real y medida por ti**.

In [ ]:
ruta_rev = RAIZ / "data" / "bases" / "revision.csv"
if not ruta_rev.exists():
    raise SystemExit("Primero ejecuta la Parte B con tus bases en data/bases/.")
rev = pd.read_csv(ruta_rev)
col = "correcto (si/no)"
rev[col] = rev[col].astype(str).str.strip().str.lower()
evaluadas = rev[rev[col].isin(["si", "sí", "no"])]
aciertos = evaluadas[col].isin(["si", "sí"]).sum()
faltaron = (rev[col] == "faltó").sum() + (rev[col] == "falto").sum()
if len(evaluadas):
    print(f"Precisión: {aciertos}/{len(evaluadas)} = {aciertos/len(evaluadas):.0%} de lo encontrado era correcto")
    print(f"Requisitos que no encontró: {faltaron}")
    print(evaluadas.groupby("categoria")[col].apply(lambda s: f"{s.isin(['si','sí']).mean():.0%}"))
else:
    print("Aún no marcaste filas con si/no en revision.csv")

## Al terminar en Colab · Descarga los resultados

Guarda estos archivos en las mismas carpetas de tu proyecto, en tu computadora:
- `articulos.json` → `data/normativa/`
- `historial.csv` → `data/ocds/`
- `revision.csv` → `data/bases/` (para marcar si/no en Excel)

In [ ]:
if EN_COLAB:
    for ruta in ["data/normativa/articulos.json", "data/ocds/historial.csv", "data/bases/revision.csv"]:
        if os.path.exists(ruta):
            files.download(ruta)
else:
    print("En tu computadora los archivos ya quedaron en la carpeta data/.")